# MAT353 – Nümerik Analiz Dönem Projesi: Canlı Google Colab Demosu
**Elektrikli Araç Bataryalarında Çok Amaçlı Optimal Şarj Stratejisinin Nümerik Optimizasyonu**

**Öğrenci:** Samet Oymak (24120205042) | **Öğretim Üyesi:** Doç. Dr. Murat GEZER

Bu defter, ham veri setleri yeniden indirilmeden (<30 saniyede) uçtan uca çalışan canlı sunum demosudur:
1. Kalibre edilmiş **LG Chem INR21700-M50T** 1-RC Thevenin + termal + yaşlanma modelini yükler.
2. **Euler, RK4 ve `solve_ivp`** doğrulama karşılaştırmasını gösterir.
3. **1.0C CC-CV** baz stratejisini ve **Gradient Descent, Adam, SLSQP** optimizasyon algoritmalarını canlı çalıştırır.
4. Örnek **Pareto** ağırlık karşılaştırması ve sabit tohumlu (`seed=2026`) **Monte Carlo** duyarlılık analizini grafikler.

In [ ]:
# 1. Ortam Kurulumu ve Kütüphaneler (Colab + Yerel Uyumlu)
import json, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Sametoymak99/EV-Battery-Optimization.git'
if IN_COLAB and REPO_URL and not (Path.cwd() / 'src').exists():
    target_dir = Path('/content/EV-Battery-Optimization')
    if not target_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(target_dir)], check=True)
    import os
    os.chdir(target_dir)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Çalışma dizini:', PROJECT_ROOT)

In [ ]:
# 2. Kalibre Edilmiş Parametrelerin Yüklenmesi (veya Gömülü Yedekten Kurulması)
EMBEDDED_CAL = {'parameters': {'Q_Ah': 4.859873076029536, 'eta': 0.998, 'R0_Ohm': 0.02865084067687651, 'R1_Ohm': 0.015848579313994565, 'C1_F': 3795.1641387721543, 'Rth_K_per_W': 4.254698318224314, 'Cth_J_per_K': 134.85830011034426, 'T_amb_C': 23.0, 'k0': 0.015, 'Ea_J_per_mol': 31500.0, 'p_crate': 1.5, 'alpha_soc': 0.5, 'Rg_J_per_mol_K': 8.314462618, 'is_calibrated': True}, 'uncertainties_std': {'Q_Ah': 0.025388805909087004, 'R0_Ohm': 0.003693054525790643, 'R1_Ohm': 0.00038112548386397553, 'C1_F': 128.81374857868806, 'Rth_K_per_W': 0.6382047477336471, 'Cth_J_per_K': 13.485830011034427}, 'provenance': {'Q_Ah': 'stanford_capacity_test_diag1_mean_W4_W8_W9_W10', 'eta': 'assumed_coulombic_efficiency_0.998', 'R0_Ohm': 'stanford_hppc_diag1_nonlinear_lsq_mean', 'R1_Ohm': 'stanford_hppc_diag1_nonlinear_lsq_mean', 'C1_F': 'stanford_hppc_diag1_nonlinear_lsq_mean', 'Rth_K_per_W': 'stanford_w10_channel_6_2_thermal_lsq', 'Cth_J_per_K': 'stanford_w10_channel_6_2_thermal_lsq', 'T_amb_C': 'stanford_chamber_23C', 'aging': 'literature_empirical_relative_indicator'}, 'thermal_fit_metrics': {'T_amb_fit_C': 22.826163962168856, 'rmse_C': 0.1620915642483151, 'mae_C': 0.12475657363671365, 'heating_rmse_C': 0.1460414277670974, 'cooling_rmse_C': 0.1737370090171809, 'r2': 0.9946684312689322}, 'per_cell_hppc_fits': [{'cell_id': 'W4', 'c_rate': 'C/4', 'Q_Ah': 4.822444927228822, 'R0_Ohm': 0.034176290797324184, 'R0_pulse_Ohm': 0.034991636643137376, 'R1_Ohm': 0.015341587188844867, 'C1_F': 3620.5889790842834, 'tau1_s': 55.54558149759236, 'rmse_V': 0.0229589830884295, 'r2': 0.9890577111362676}, {'cell_id': 'W8', 'c_rate': 'C/2', 'Q_Ah': 4.876858882134159, 'R0_Ohm': 0.026847622208043806, 'R0_pulse_Ohm': 0.026888246548696107, 'R1_Ohm': 0.016128757735941787, 'C1_F': 3864.9106472676194, 'tau1_s': 62.3362075008414, 'rmse_V': 0.017790961049091145, 'r2': 0.9932935675607687}, {'cell_id': 'W9', 'c_rate': '1C', 'Q_Ah': 4.874304103511904, 'R0_Ohm': 0.02711085961822415, 'R0_pulse_Ohm': 0.027055920713611906, 'R1_Ohm': 0.016154997450606286, 'C1_F': 3914.449087910826, 'tau1_s': 63.2379150357275, 'rmse_V': 0.017472613090055798, 'r2': 0.99353825379328}, {'cell_id': 'W10', 'c_rate': '3C', 'Q_Ah': 4.865884391243259, 'R0_Ohm': 0.026468590083913907, 'R0_pulse_Ohm': 0.026787858732082382, 'R1_Ohm': 0.01576897488058532, 'C1_F': 3780.707840825887, 'tau1_s': 59.617886972815384, 'rmse_V': 0.0185958745496435, 'r2': 0.9926333913296245}], 'eis_r0_summary': [{'cell_id': 'W8', 'diag_number': 1, 'soc_pct': 20, 'freq_Hz': 2.1701390743255615, 'R0_eis_Ohm': 0.026283452562387736, 'im_z_Ohm': -0.0003512293347790237}, {'cell_id': 'W8', 'diag_number': 1, 'soc_pct': 50, 'freq_Hz': 4.650298118591309, 'R0_eis_Ohm': 0.025169467800439216, 'im_z_Ohm': -0.000277772100719074}, {'cell_id': 'W8', 'diag_number': 1, 'soc_pct': 80, 'freq_Hz': 9.973400115966797, 'R0_eis_Ohm': 0.025072219570068183, 'im_z_Ohm': -0.0003382095451422882}, {'cell_id': 'W9', 'diag_number': 1, 'soc_pct': 20, 'freq_Hz': 2.1701390743255615, 'R0_eis_Ohm': 0.02643779095982211, 'im_z_Ohm': -0.0003568709717022197}, {'cell_id': 'W9', 'diag_number': 1, 'soc_pct': 50, 'freq_Hz': 4.650298118591309, 'R0_eis_Ohm': 0.02539328463562064, 'im_z_Ohm': -0.00026815527738874686}, {'cell_id': 'W9', 'diag_number': 1, 'soc_pct': 80, 'freq_Hz': 4.650298118591309, 'R0_eis_Ohm': 0.025274305927194297, 'im_z_Ohm': -0.00029260849289489756}, {'cell_id': 'W10', 'diag_number': 1, 'soc_pct': 20, 'freq_Hz': 4.650298118591309, 'R0_eis_Ohm': 0.02612859225812288, 'im_z_Ohm': -0.00033960808932414176}, {'cell_id': 'W10', 'diag_number': 1, 'soc_pct': 50, 'freq_Hz': 4.650298118591309, 'R0_eis_Ohm': 0.02508868112107822, 'im_z_Ohm': -0.00026685172502264476}, {'cell_id': 'W10', 'diag_number': 1, 'soc_pct': 80, 'freq_Hz': 4.650298118591309, 'R0_eis_Ohm': 0.025060094978899657, 'im_z_Ohm': -0.00025581251828280384}], 'ocv_soc_grid': [0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045, 0.05, 0.055, 0.06, 0.065, 0.07, 0.075, 0.08, 0.085, 0.09, 0.095, 0.1, 0.105, 0.11, 0.115, 0.12, 0.125, 0.13, 0.135, 0.14, 0.145, 0.15, 0.155, 0.16, 0.165, 0.17, 0.17500000000000002, 0.18, 0.185, 0.19, 0.195, 0.2, 0.20500000000000002, 0.21, 0.215, 0.22, 0.225, 0.23, 0.23500000000000001, 0.24, 0.245, 0.25, 0.255, 0.26, 0.265, 0.27, 0.275, 0.28, 0.28500000000000003, 0.29, 0.295, 0.3, 0.305, 0.31, 0.315, 0.32, 0.325, 0.33, 0.335, 0.34, 0.34500000000000003, 0.35000000000000003, 0.355, 0.36, 0.365, 0.37, 0.375, 0.38, 0.385, 0.39, 0.395, 0.4, 0.405, 0.41000000000000003, 0.41500000000000004, 0.42, 0.425, 0.43, 0.435, 0.44, 0.445, 0.45, 0.455, 0.46, 0.465, 0.47000000000000003, 0.47500000000000003, 0.48, 0.485, 0.49, 0.495, 0.5, 0.505, 0.51, 0.515, 0.52, 0.525, 0.53, 0.535, 0.54, 0.545, 0.55, 0.555, 0.56, 0.5650000000000001, 0.5700000000000001, 0.5750000000000001, 0.58, 0.585, 0.59, 0.595, 0.6, 0.605, 0.61, 0.615, 0.62, 0.625, 0.63, 0.635, 0.64, 0.645, 0.65, 0.655, 0.66, 0.665, 0.67, 0.675, 0.68, 0.685, 0.6900000000000001, 0.6950000000000001, 0.7000000000000001, 0.705, 0.71, 0.715, 0.72, 0.725, 0.73, 0.735, 0.74, 0.745, 0.75, 0.755, 0.76, 0.765, 0.77, 0.775, 0.78, 0.785, 0.79, 0.795, 0.8, 0.805, 0.81, 0.8150000000000001, 0.8200000000000001, 0.8250000000000001, 0.8300000000000001, 0.835, 0.84, 0.845, 0.85, 0.855, 0.86, 0.865, 0.87, 0.875, 0.88, 0.885, 0.89, 0.895, 0.9, 0.905, 0.91, 0.915, 0.92, 0.925, 0.93, 0.935, 0.9400000000000001, 0.9450000000000001, 0.9500000000000001, 0.9550000000000001, 0.96, 0.965, 0.97, 0.975, 0.98, 0.985, 0.99, 0.995, 1.0], 'ocv_V_grid': [2.5103204250335693, 2.6037225247280755, 2.6926491503769707, 2.7664396421666337, 2.826250991129714, 2.873643921892908, 2.912450044113644, 2.9457164587795557, 2.974912356997113, 3.000937310471309, 3.0241373065135764, 3.0449791987212453, 3.064170505831286, 3.0821862116967553, 3.099347863056076, 3.1161368261402074, 3.132873699696352, 3.1495131091185105, 3.166214015804738, 3.182925482970756, 3.1996625240843795, 3.216137041667179, 3.2323713487196835, 3.2481602974283943, 3.2634348188295457, 3.2781146680935684, 3.292024767243381, 3.3053261179842535, 3.318093079441, 3.330426539890622, 3.342490585557359, 3.3545064319194267, 3.3662198761266877, 3.377547816399689, 3.389512925303712, 3.402277191102625, 3.413257382638884, 3.422273999962304, 3.429604752740496, 3.43541115843661, 3.4403712700759135, 3.4447457762103486, 3.4488524316760647, 3.452743122071038, 3.4563856155215036, 3.460026021195563, 3.4634737613939155, 3.466981542619319, 3.470534779568009, 3.4745498589263466, 3.479253506771183, 3.484829907287959, 3.4908678915741813, 3.4970848183574264, 3.503269176609062, 3.5094514743136433, 3.515573561103597, 3.5217541735369857, 3.5276827532238406, 3.533475913231991, 3.5388861596199543, 3.543969821487333, 3.548895186371819, 3.5535337195928625, 3.5580961018625374, 3.5625586231335937, 3.567014162834676, 3.571489622102316, 3.5758823498133188, 3.580146622699276, 3.5842728877172423, 3.5882426950797237, 3.5919899886449995, 3.595605664449062, 3.5991468388936685, 3.6024953891416054, 3.6057954667369145, 3.609053071434487, 3.6123008673461907, 3.615479604684879, 3.618722938185267, 3.621817053099392, 3.625025642814591, 3.6282401179761243, 3.631528111322869, 3.6348337268718036, 3.6381714464259063, 3.6416544611793547, 3.645097821397128, 3.648673461569592, 3.6522492393185666, 3.6560045509683445, 3.6597813404381823, 3.6637209101405763, 3.6677225748432107, 3.6718010660701816, 3.67597446651409, 3.680290597094584, 3.6847188403162563, 3.6892089971601045, 3.693705546032143, 3.6984069949020735, 3.703119809452856, 3.707958474311651, 3.712804929528372, 3.7176648095590865, 3.7225506523809635, 3.7274979520259506, 3.732463921096347, 3.737470461763434, 3.7424281121106078, 3.7473938212737306, 3.752370876620518, 3.757285950905123, 3.7622359572153745, 3.767239715728394, 3.7721792033151016, 3.7770507913449753, 3.7819730120593493, 3.786931202258142, 3.7919338070562834, 3.796968268636921, 3.8020388305949493, 3.80719997048056, 3.812489746574059, 3.8178945328813647, 3.8234477926220194, 3.8290502655662504, 3.8347402619693716, 3.840394642736351, 3.8460589498349154, 3.8516879620369098, 3.857246963348366, 3.8626153605587747, 3.8678555233179095, 3.8728804860253505, 3.877834195006175, 3.8826394933046267, 3.887332783869312, 3.8918731142233653, 3.8964516761045087, 3.900982160675516, 3.905472286474523, 3.9100643837937294, 3.914653648696031, 3.9193486414109997, 3.9240503119842445, 3.928776989018508, 3.93372418395859, 3.938739346902638, 3.943844264812044, 3.9490637373385677, 3.954408549813726, 3.959864907570328, 3.9653570842336383, 3.9709819420260697, 3.9766533749705655, 3.9823780838945653, 3.9881374035154415, 3.9938563816035346, 3.999533748761775, 4.005143287135947, 4.010771068215004, 4.016198233774924, 4.0215327155933664, 4.0266454096555595, 4.031677364460631, 4.036430627612717, 4.040942232244388, 4.045232106493786, 4.049196166495622, 4.052850140557355, 4.056104308283531, 4.059046283321994, 4.061613134038112, 4.063991630957458, 4.066039656012949, 4.067982984619417, 4.069775334956652, 4.071589964745173, 4.07333439191155, 4.075122922005396, 4.076906935230018, 4.078793635034652, 4.080725070549228, 4.082834354445289, 4.085191127535218, 4.087678178654509, 4.090447156974166, 4.093515073354439, 4.0968570026638655, 4.100699658514788, 4.105068281082717, 4.110038098295265, 4.115762742766849, 4.12232882434212, 4.129890684861857, 4.138656483453005, 4.148955680574853, 4.1612124935948955, 4.189442873001099]}
EMBEDDED_REF = {'baseline_strategy': 'CC-CV 1.0C (SOC 0.20 -> 0.90)', 't_ref_s': 2688.8703959776926, 'E_ref_J': 2518.923678881762, 'D_ref': 0.00013879829876334258, 'i_cc_ref_A': 4.859873076029536}

cal_path = PROJECT_ROOT / 'data' / 'processed' / 'calibrated_parameters.json'
ref_path = PROJECT_ROOT / 'data' / 'processed' / 'reference_scales.json'
if not cal_path.exists():
    cal_path.parent.mkdir(parents=True, exist_ok=True)
    cal_path.write_text(json.dumps(EMBEDDED_CAL, indent=2), encoding='utf-8')
if not ref_path.exists():
    ref_path.write_text(json.dumps(EMBEDDED_REF, indent=2), encoding='utf-8')

from src.battery_model import BatteryLimits, BatteryParameters, OCVInterpolator, simulate_soc_profile
from src.cc_cv import simulate_cc_cv
from src.objective_functions import ReferenceScales, evaluate_objectives
from src.optimizers import optimize_gradient_descent, optimize_adam, optimize_slsqp
from src.evaluation import sample_uncertain_parameters

params = BatteryParameters.from_calibrated_json(cal_path)
ocv_fn = OCVInterpolator.from_calibrated_json(cal_path)
limits = BatteryLimits()
ref_scales = ReferenceScales.from_json(ref_path)
print(f'Kalibre Model Yüklendi: Q={params.Q_Ah:.4f} Ah, R0={params.R0_Ohm:.5f} Ohm, R1={params.R1_Ohm:.5f} Ohm, C1={params.C1_F:.1f} F, Rth={params.Rth_K_per_W:.2f} K/W, Cth={params.Cth_J_per_K:.1f} J/K')

In [ ]:
# 3. Canlı ODE Çözücü Kontrolü (Euler vs RK4 vs solve_ivp)
u_test = np.array([6.0, 5.5, 5.0, 4.5, 4.0, 3.5, 3.0, 2.5, 2.0, 1.5])
for m in ('euler', 'rk4', 'solve_ivp'):
    s = simulate_soc_profile(u_test, params, ocv_fn, method=m, dt_max_s=5.0)
    print(f"{m:9s}: tf={s['final_time_s']/60:.2f} dk, Vmax={s['max_voltage_V']:.5f} V, Tmax={s['max_temperature_C']:.2f} C, süre={s['runtime_s']*1000:.1f} ms")

In [ ]:
# 4. CC-CV (1.0C) vs. Gradient Descent, Adam ve SLSQP Canlı Optimizasyon
weights = (0.4, 0.2, 0.4)
cccv = simulate_cc_cv(params, ocv_fn, limits, i_cc_A=1.0*params.Q_Ah, dt_s=1.0)
cccv_obj = evaluate_objectives(cccv, ref_scales, weights=weights)

res_slsqp = optimize_slsqp(params, ocv_fn, limits, ref_scales, weights=weights, dt_opt_s=10.0)
res_adam = optimize_adam(params, ocv_fn, limits, ref_scales, weights=weights, max_iters_per_stage=25, dt_opt_s=10.0)
res_gd = optimize_gradient_descent(params, ocv_fn, limits, ref_scales, weights=weights, max_iters_per_stage=25, dt_opt_s=10.0)

rows = [{'Strateji': 'CC-CV (1.0C)', 'Süre [dk]': cccv['final_time_s']/60, 'Kayıp [Wh]': cccv['total_energy_loss_J']/3600, 'Yaşlanma D': cccv['total_aging_D'], 'J_total': cccv_obj['J_total'], 'Vmax [V]': cccv['max_voltage_V'], 'Tmax [°C]': cccv['max_temperature_C'], 'Hesap [s]': cccv['runtime_s']}]
for r in (res_gd, res_adam, res_slsqp):
    s = r['simulation']
    rows.append({'Strateji': r['algorithm'], 'Süre [dk]': s['final_time_s']/60, 'Kayıp [Wh]': s['total_energy_loss_J']/3600, 'Yaşlanma D': s['total_aging_D'], 'J_total': r['objectives']['J_total'], 'Vmax [V]': s['max_voltage_V'], 'Tmax [°C]': s['max_temperature_C'], 'Hesap [s]': r['runtime_s']})
df_cmp = pd.DataFrame(rows)
display(df_cmp.round(5))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for lbl, sim, ls in [('CC-CV 1C', cccv, '--'), ('GD', res_gd['simulation'], '-'), ('Adam', res_adam['simulation'], '-'), ('SLSQP', res_slsqp['simulation'], '-')]:
    t_m = sim['time_s'] / 60.0
    axes[0].plot(t_m, sim['current_A'], ls, label=lbl)
    axes[1].plot(t_m, sim['terminal_voltage_V'], ls, label=lbl)
    axes[2].plot(t_m, sim['temperature_C'], ls, label=lbl)
axes[0].set_title('Akım [A]'); axes[0].set_xlabel('Süre [dk]'); axes[0].set_ylabel('Akım [A]'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Gerilim [V]'); axes[1].axhline(4.20, c='k', ls=':'); axes[1].set_xlabel('Süre [dk]'); axes[1].set_ylabel('Gerilim [V]'); axes[1].grid(True, alpha=0.3)
axes[2].set_title('Sıcaklık [°C]'); axes[2].set_xlabel('Süre [dk]'); axes[2].set_ylabel('Sıcaklık [°C]'); axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 5. Sabit Seed (seed=2026) ile Hızlı Monte Carlo Duyarlılık Kontrolü (N=30)
stds = EMBEDDED_CAL['uncertainties_std']
samples = sample_uncertain_parameters(params, stds, n_samples=30, seed=2026)
mc_eloss, mc_vmax = [], []
for _, row in samples.iterrows():
    p_i = BatteryParameters(Q_Ah=row['Q_Ah'], eta=params.eta, R0_Ohm=row['R0_Ohm'], R1_Ohm=row['R1_Ohm'], C1_F=row['C1_F'], Rth_K_per_W=row['Rth_K_per_W'], Cth_J_per_K=row['Cth_J_per_K'], T_amb_C=params.T_amb_C, is_calibrated=True)
    sim_i = simulate_soc_profile(res_slsqp['u_opt_A'], p_i, ocv_fn, method='rk4', dt_max_s=5.0)
    mc_eloss.append(sim_i['total_energy_loss_J'] / 3600.0)
    mc_vmax.append(sim_i['max_voltage_V'])
print(f'Monte Carlo (N=30, seed=2026): Enerji Kaybı = {np.mean(mc_eloss):.4f} ± {np.std(mc_eloss, ddof=1):.4f} Wh | Vmax = {np.mean(mc_vmax):.4f} ± {np.std(mc_vmax, ddof=1):.4f} V')